# Chapter 8.8: Performance Tuning Your Model Implementation

## Learning Objectives

By the end of this notebook, you will:

1. **Apply** profiling methodology to identify bottlenecks
2. **Identify** bottlenecks: attention, FFN, communication
3. **Select** and tune kernels for optimal performance
4. **Optimize** memory usage to reduce peak consumption
5. **Ensure** CUDA graph compatibility
6. **Understand** batch size and sequence length impact
7. **Apply** throughput and latency optimization techniques
8. **Conduct** A/B performance testing

---

## Prerequisites
- Chapters 8.1-8.7 (Full model implementation and testing)
- Basic understanding of GPU programming concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.8_performance.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.8_performance.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Performance Profiling Methodology

```
Performance Tuning Workflow
===========================

1. MEASURE (Establish baseline)
   |
   v
2. PROFILE (Find bottlenecks)
   |
   v
3. OPTIMIZE (Fix top bottleneck)
   |
   v
4. VALIDATE (Correctness check)
   |
   v
5. MEASURE (Check improvement)
   |
   +--- If improved, go to step 2
   +--- If not improved, revert and try different approach
```

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np
from typing import Dict, List, Tuple, Optional

# Profiling tools overview
print("Profiling Tools for vLLM/SGLang Models")
print("=" * 60)

tools = {
    "torch.profiler": {
        "use_case": "Detailed kernel-level profiling",
        "output": "Chrome trace (JSON), TensorBoard",
        "overhead": "Low-medium",
        "gpu_support": "Yes (CUDA events)",
    },
    "torch.cuda.Event": {
        "use_case": "Quick GPU timing of specific operations",
        "output": "Elapsed time in milliseconds",
        "overhead": "Very low",
        "gpu_support": "Yes",
    },
    "nsight systems (nsys)": {
        "use_case": "System-level profiling (CPU + GPU)",
        "output": "NVIDIA Nsight timeline",
        "overhead": "Low",
        "gpu_support": "Yes (detailed)",
    },
    "nsight compute (ncu)": {
        "use_case": "Deep kernel analysis (occupancy, memory, etc.)",
        "output": "NVIDIA Nsight Compute report",
        "overhead": "Very high (replays kernels)",
        "gpu_support": "Yes (very detailed)",
    },
    "torch.cuda.memory_stats": {
        "use_case": "Memory usage analysis",
        "output": "Memory allocation statistics",
        "overhead": "None",
        "gpu_support": "Yes",
    },
}

for name, info in tools.items():
    print(f"\n{name}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

In [ ]:
# Simple profiling utilities

class Timer:
    """Simple CPU timer."""
    def __init__(self, name=""):
        self.name = name
    
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        self.elapsed = (time.perf_counter() - self.start) * 1000  # ms
        if self.name:
            print(f"  {self.name}: {self.elapsed:.2f} ms")


class ModelProfiler:
    """
    Profile a model's forward pass to identify bottlenecks.
    
    Works on CPU for tutorial; on GPU, use CUDA events instead.
    """
    
    def __init__(self, model: nn.Module):
        self.model = model
        self.timings: Dict[str, List[float]] = {}
        self._hooks = []
    
    def _create_hook(self, name: str):
        def hook(module, input, output):
            # Record timing (simplified - on GPU use CUDA events)
            if name not in self.timings:
                self.timings[name] = []
        return hook
    
    def profile_forward(self, input_ids: torch.Tensor, n_warmup: int = 5, n_runs: int = 20):
        """
        Profile a forward pass.
        """
        print(f"Profiling model with input shape {input_ids.shape}")
        print(f"  Warmup: {n_warmup} iterations")
        print(f"  Measurement: {n_runs} iterations")
        
        # Warmup
        for _ in range(n_warmup):
            with torch.no_grad():
                _ = self.model(input_ids)
        
        # Measure
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            with torch.no_grad():
                _ = self.model(input_ids)
            elapsed = (time.perf_counter() - start) * 1000
            times.append(elapsed)
        
        times = np.array(times)
        print(f"\nResults:")
        print(f"  Mean:   {times.mean():.2f} ms")
        print(f"  Median: {np.median(times):.2f} ms")
        print(f"  P95:    {np.percentile(times, 95):.2f} ms")
        print(f"  P99:    {np.percentile(times, 99):.2f} ms")
        print(f"  Std:    {times.std():.2f} ms")
        
        return times

# Demo
model = nn.Sequential(
    nn.Embedding(1000, 256),
    nn.Linear(256, 1024),
    nn.ReLU(),
    nn.Linear(1024, 256),
    nn.LayerNorm(256),
    nn.Linear(256, 1000),
)

profiler = ModelProfiler(model)
input_ids = torch.randint(0, 1000, (1, 128))
times = profiler.profile_forward(input_ids)

## 2. Identifying Bottlenecks

In a typical transformer, the compute is distributed across several components.

In [ ]:
# Theoretical compute breakdown for a Llama-7B model

def compute_breakdown(hidden_size, num_heads, num_kv_heads, intermediate_size,
                      num_layers, vocab_size, seq_len, batch_size):
    """
    Estimate FLOPs for each component of a transformer forward pass.
    """
    head_dim = hidden_size // num_heads
    
    # Per-layer FLOPs
    qkv_flops = 2 * seq_len * batch_size * hidden_size * (num_heads + 2 * num_kv_heads) * head_dim
    attn_flops = 2 * seq_len * batch_size * seq_len * num_heads * head_dim  # Q*K and attn*V
    o_proj_flops = 2 * seq_len * batch_size * hidden_size * hidden_size
    
    gate_up_flops = 2 * seq_len * batch_size * hidden_size * 2 * intermediate_size
    down_proj_flops = 2 * seq_len * batch_size * intermediate_size * hidden_size
    
    # Total per layer
    attention_total = qkv_flops + attn_flops + o_proj_flops
    ffn_total = gate_up_flops + down_proj_flops
    
    # Model-wide
    embed_flops = seq_len * batch_size * hidden_size  # Lookup
    lm_head_flops = 2 * seq_len * batch_size * hidden_size * vocab_size
    
    total = num_layers * (attention_total + ffn_total) + embed_flops + lm_head_flops
    
    return {
        "QKV projection": qkv_flops * num_layers,
        "Attention (Q*K, A*V)": attn_flops * num_layers,
        "O projection": o_proj_flops * num_layers,
        "Gate+Up projection": gate_up_flops * num_layers,
        "Down projection": down_proj_flops * num_layers,
        "LM Head": lm_head_flops,
        "Total": total,
    }

# Llama-7B parameters
breakdown = compute_breakdown(
    hidden_size=4096, num_heads=32, num_kv_heads=32,
    intermediate_size=11008, num_layers=32,
    vocab_size=32000, seq_len=2048, batch_size=1
)

print("Compute Breakdown: Llama-7B (seq_len=2048, batch=1)")
print("=" * 60)
total = breakdown["Total"]
for component, flops in breakdown.items():
    if component == "Total":
        print(f"\n{'Total':30s} {flops/1e12:.1f} TFLOP")
    else:
        pct = flops / total * 100
        bar = '#' * int(pct / 2)
        print(f"  {component:28s} {flops/1e12:6.1f} TFLOP ({pct:5.1f}%) {bar}")

print("\nKey insight: FFN (gate+up+down) dominates compute, not attention!")
print("Attention compute grows quadratically with seq_len.")

In [ ]:
# Compute breakdown at different sequence lengths

print("How Compute Distribution Changes with Sequence Length")
print("=" * 70)

for seq_len in [128, 512, 2048, 8192, 32768]:
    bd = compute_breakdown(
        hidden_size=4096, num_heads=32, num_kv_heads=32,
        intermediate_size=11008, num_layers=32,
        vocab_size=32000, seq_len=seq_len, batch_size=1
    )
    total = bd["Total"]
    attn_pct = (bd["QKV projection"] + bd["Attention (Q*K, A*V)"] + bd["O projection"]) / total * 100
    ffn_pct = (bd["Gate+Up projection"] + bd["Down projection"]) / total * 100
    head_pct = bd["LM Head"] / total * 100
    
    print(f"  seq_len={seq_len:>6}: Attention={attn_pct:5.1f}%, FFN={ffn_pct:5.1f}%, LMHead={head_pct:5.1f}%, Total={total/1e12:.1f} TFLOP")

print("\nAt short sequences: FFN dominates (memory-bandwidth bound)")
print("At long sequences: Attention grows quadratically (compute-bound)")

## 3. Kernel Selection and Tuning

In [ ]:
# Kernel selection guide

print("Kernel Selection Guide")
print("=" * 60)

kernels = {
    "Attention Kernels": {
        "FlashAttention-2": {
            "best_for": "General-purpose attention, prefill",
            "gpu": "Ampere+ (A100, H100)",
            "features": "IO-aware, tiling, supports GQA",
        },
        "FlashInfer": {
            "best_for": "Decode attention, paged KV cache",
            "gpu": "Ampere+ (A100, H100)",
            "features": "Optimized for decode, RadixAttention support",
        },
        "xformers": {
            "best_for": "Fallback when FlashAttention unavailable",
            "gpu": "Turing+ (T4, V100)",
            "features": "Memory-efficient attention",
        },
    },
    "Linear Kernels": {
        "cuBLAS": {
            "best_for": "FP16/BF16 matrix multiplication",
            "gpu": "All NVIDIA GPUs",
            "features": "Default, highly optimized by NVIDIA",
        },
        "CUTLASS": {
            "best_for": "Custom GEMM configurations",
            "gpu": "Ampere+",
            "features": "Template-based, flexible",
        },
        "Marlin": {
            "best_for": "INT4 quantized matmul",
            "gpu": "Ampere+ (A100, H100)",
            "features": "Near-FP16 speed for INT4, GPTQ/AWQ",
        },
    },
    "Activation Kernels": {
        "SiluAndMul": {
            "best_for": "Fused SiLU + element-wise multiply",
            "gpu": "All",
            "features": "Avoids intermediate memory allocation",
        },
        "FusedRMSNorm": {
            "best_for": "Fused RMSNorm + residual add",
            "gpu": "All",
            "features": "Reduces memory traffic",
        },
    },
}

for category, kernel_dict in kernels.items():
    print(f"\n{category}:")
    for name, info in kernel_dict.items():
        print(f"  {name}:")
        print(f"    Best for: {info['best_for']}")
        print(f"    GPU: {info['gpu']}")
        print(f"    Features: {info['features']}")

## 4. Memory Optimization

In [ ]:
def estimate_model_memory(hidden_size, num_layers, vocab_size, intermediate_size,
                           num_heads, num_kv_heads, dtype_bytes=2):
    """
    Estimate model weight memory.
    """
    head_dim = hidden_size // num_heads
    
    # Per layer
    qkv_params = hidden_size * (num_heads + 2 * num_kv_heads) * head_dim
    o_proj_params = hidden_size * hidden_size
    gate_up_params = hidden_size * 2 * intermediate_size
    down_proj_params = intermediate_size * hidden_size
    norm_params = 2 * hidden_size  # input_norm + post_attn_norm
    
    per_layer = qkv_params + o_proj_params + gate_up_params + down_proj_params + norm_params
    
    # Global
    embed_params = vocab_size * hidden_size
    lm_head_params = vocab_size * hidden_size
    final_norm_params = hidden_size
    
    total_params = num_layers * per_layer + embed_params + lm_head_params + final_norm_params
    total_bytes = total_params * dtype_bytes
    
    return {
        "total_params": total_params,
        "total_bytes": total_bytes,
        "per_layer_bytes": per_layer * dtype_bytes,
        "embedding_bytes": embed_params * dtype_bytes,
        "lm_head_bytes": lm_head_params * dtype_bytes,
    }

def estimate_kv_cache_memory(num_layers, num_kv_heads, head_dim, max_seq_len,
                              max_batch_size, dtype_bytes=2):
    """
    Estimate KV cache memory.
    """
    # Per token: K + V for all layers
    per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes
    total = per_token * max_seq_len * max_batch_size
    
    return {
        "per_token_bytes": per_token,
        "total_bytes": total,
    }

# Llama-7B memory breakdown
model_mem = estimate_model_memory(
    hidden_size=4096, num_layers=32, vocab_size=32000,
    intermediate_size=11008, num_heads=32, num_kv_heads=32, dtype_bytes=2  # FP16
)

kv_mem = estimate_kv_cache_memory(
    num_layers=32, num_kv_heads=32, head_dim=128,
    max_seq_len=4096, max_batch_size=32, dtype_bytes=2
)

print("Memory Breakdown: Llama-7B (FP16)")
print("=" * 50)
print(f"Model weights:")
print(f"  Total params: {model_mem['total_params']/1e9:.2f}B")
print(f"  Weight memory: {model_mem['total_bytes']/1e9:.2f} GB")
print(f"  Per layer: {model_mem['per_layer_bytes']/1e6:.0f} MB")
print(f"  Embeddings: {model_mem['embedding_bytes']/1e6:.0f} MB")
print(f"  LM Head: {model_mem['lm_head_bytes']/1e6:.0f} MB")

print(f"\nKV Cache (max_seq=4096, batch=32):")
print(f"  Per token: {kv_mem['per_token_bytes']} bytes")
print(f"  Total: {kv_mem['total_bytes']/1e9:.2f} GB")

total_gpu = model_mem['total_bytes'] + kv_mem['total_bytes']
print(f"\nTotal GPU memory needed: {total_gpu/1e9:.2f} GB")
print(f"  (weights + KV cache, excludes activations)")

In [ ]:
# Memory optimization techniques

print("Memory Optimization Techniques")
print("=" * 60)

techniques = [
    {
        "technique": "Fused Operations",
        "savings": "10-30% activation memory",
        "description": "Fuse add+norm, SiLU+mul to avoid intermediate tensors",
        "example": "RMSNorm(x, residual) fuses x=norm(x+residual)",
    },
    {
        "technique": "In-place Operations",
        "savings": "Up to 50% activation memory",
        "description": "Modify tensors in-place instead of creating new ones",
        "example": "x.add_(residual) instead of x = x + residual",
    },
    {
        "technique": "Quantization",
        "savings": "2-4x weight memory",
        "description": "Store weights in INT4/INT8/FP8",
        "example": "GPTQ/AWQ for 4x compression",
    },
    {
        "technique": "Grouped Query Attention (GQA)",
        "savings": "Up to 8x KV cache memory",
        "description": "Use fewer KV heads (8 vs 32)",
        "example": "Llama-3: 32 Q heads, 8 KV heads -> 4x reduction",
    },
    {
        "technique": "Paged KV Cache",
        "savings": "Eliminate fragmentation waste",
        "description": "Allocate KV cache in pages, not contiguous blocks",
        "example": "vLLM PagedAttention: ~2x more sequences in same memory",
    },
    {
        "technique": "Tensor Parallelism",
        "savings": "Linear scaling with # GPUs",
        "description": "Split model across GPUs",
        "example": "TP=4: each GPU holds 25% of weights",
    },
]

for t in techniques:
    print(f"\n{t['technique']}:")
    print(f"  Savings: {t['savings']}")
    print(f"  Description: {t['description']}")
    print(f"  Example: {t['example']}")

## 5. CUDA Graph Compatibility

In [ ]:
# CUDA Graph overview and compatibility

print("CUDA Graphs for LLM Inference")
print("=" * 60)
print("""
What are CUDA Graphs?
  CUDA Graphs capture a sequence of GPU operations (kernels, memory copies)
  and replay them with minimal CPU overhead. This eliminates:
  - Kernel launch overhead (~5-10 us per kernel)
  - CPU-GPU synchronization overhead
  - Python overhead

Why they matter for LLM inference:
  - Decode phase launches hundreds of small kernels per step
  - Without graphs: ~1-2ms CPU overhead per decode step
  - With graphs: ~0.01ms overhead per decode step
  - Result: 20-50% latency reduction for decode

Compatibility requirements:
  1. Fixed tensor shapes (no dynamic shapes)
  2. No CPU-GPU synchronization during graph
  3. No Python control flow (if/else) based on tensor values
  4. All tensors must be pre-allocated
  5. No torch.autograd (inference only)

Common issues with CUDA graph compatibility:
""")

issues = [
    ("Dynamic shapes in attention",
     "Different batch sizes need different graph captures",
     "Pre-capture graphs for common batch sizes"),
    ("Conditional operations",
     "if seq_len > threshold: use_chunked_attention()",
     "Use padding and masks instead of branching"),
    ("Dynamic memory allocation",
     "Creating new tensors inside the graph",
     "Pre-allocate all buffers"),
    ("CPU-GPU sync",
     "Calling .item() or .cpu() inside the graph",
     "Keep all operations on GPU"),
    ("Custom ops without graph support",
     "Third-party CUDA kernels may not support graphs",
     "Use vLLM/SGLang's built-in kernels"),
]

for issue, symptom, solution in issues:
    print(f"  Issue: {issue}")
    print(f"    Symptom: {symptom}")
    print(f"    Fix: {solution}")
    print()

In [ ]:
# CUDA Graph capture example (CPU simulation)

CUDA_GRAPH_CODE = '''
# How vLLM captures CUDA graphs for decode

import torch

# Step 1: Capture the graph
def capture_cuda_graph(model, input_example, pool=None):
    """
    Capture a CUDA graph for the model's forward pass.
    """
    # Warmup
    for _ in range(3):
        model(input_example)
    torch.cuda.synchronize()
    
    # Capture
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph, pool=pool):
        output = model(input_example)
    
    return graph, input_example, output

# Step 2: Replay the graph
def replay_cuda_graph(graph, input_buffer, output_buffer, new_input):
    """
    Replay the captured graph with new input.
    """
    # Copy new input to the captured input buffer
    input_buffer.copy_(new_input)
    # Replay the graph (very fast!)
    graph.replay()
    # Output is automatically written to output_buffer
    return output_buffer

# Step 3: vLLM pre-captures graphs for different batch sizes
# This is done during warmup:
batch_sizes_to_capture = [1, 2, 4, 8, 16, 32, 64, 128, 256]
graphs = {}
for bs in batch_sizes_to_capture:
    input_ex = torch.zeros(bs, dtype=torch.long, device="cuda")
    graphs[bs] = capture_cuda_graph(model, input_ex)
'''

print(CUDA_GRAPH_CODE)

## 6. Batch Size and Sequence Length Impact

In [ ]:
# Performance characteristics at different operating points

print("Performance at Different Operating Points")
print("=" * 70)

# Decode performance (one token per sequence)
print("\nDECODE PHASE (generating tokens):")
print(f"{'Batch Size':>12} {'Tokens/s':>12} {'Latency':>12} {'GPU Util':>12} {'Bottleneck':>15}")
print("-" * 63)

decode_data = [
    (1, 50, "20ms", "5%", "Memory BW"),
    (4, 180, "22ms", "15%", "Memory BW"),
    (16, 600, "27ms", "40%", "Memory BW"),
    (64, 1800, "36ms", "70%", "Compute"),
    (256, 4000, "64ms", "85%", "Compute"),
    (512, 5000, "102ms", "90%", "Compute+Mem"),
]

for bs, tps, lat, util, bottleneck in decode_data:
    print(f"{bs:>12} {tps:>12} {lat:>12} {util:>12} {bottleneck:>15}")

print("\nKey insights:")
print("  - Small batch: Memory bandwidth bound (weights are large, compute is small)")
print("  - Large batch: Compute bound (enough parallelism to saturate ALUs)")
print("  - Sweet spot: where throughput/latency ratio is best for your use case")

# Prefill performance
print("\nPREFILL PHASE (processing prompt):")
print(f"{'Seq Length':>12} {'Tokens/s':>12} {'Latency':>12} {'GPU Util':>12} {'Bottleneck':>15}")
print("-" * 63)

prefill_data = [
    (32, 8000, "4ms", "30%", "Kernel launch"),
    (128, 25000, "5ms", "60%", "Compute"),
    (512, 40000, "13ms", "80%", "Compute"),
    (2048, 30000, "68ms", "85%", "Attention"),
    (8192, 15000, "546ms", "90%", "Attention"),
    (32768, 5000, "6.5s", "95%", "Attention+Mem"),
]

for sl, tps, lat, util, bottleneck in prefill_data:
    print(f"{sl:>12} {tps:>12} {lat:>12} {util:>12} {bottleneck:>15}")

## 7. Throughput Optimization Techniques

In [ ]:
# Throughput optimization techniques

print("Throughput Optimization Techniques")
print("=" * 60)

throughput_techniques = [
    {
        "technique": "Continuous Batching",
        "impact": "2-5x throughput improvement",
        "description": (
            "Instead of waiting for all sequences to finish, "
            "new sequences start as soon as old ones finish. "
            "Keeps the GPU busy at all times."
        ),
    },
    {
        "technique": "Speculative Decoding",
        "impact": "1.5-3x decode speedup",
        "description": (
            "Use a small draft model to predict multiple tokens, "
            "verify them in parallel with the large model. "
            "Amortizes the cost of loading model weights."
        ),
    },
    {
        "technique": "Chunked Prefill",
        "impact": "Better decode latency during prefill",
        "description": (
            "Break long prefills into chunks and interleave "
            "with decode steps. Prevents long prefills from "
            "blocking decode."
        ),
    },
    {
        "technique": "Prefix Caching",
        "impact": "Up to 10x speedup for repeated prefixes",
        "description": (
            "Cache KV pairs for common prefixes (system prompts). "
            "Skip prefill computation for cached portions. "
            "SGLang's RadixCache excels at this."
        ),
    },
    {
        "technique": "Multi-step Scheduling",
        "impact": "Reduced scheduling overhead",
        "description": (
            "Run multiple decode steps before returning to the "
            "scheduler. Reduces Python overhead between steps."
        ),
    },
]

for t in throughput_techniques:
    print(f"\n{t['technique']}:")
    print(f"  Impact: {t['impact']}")
    print(f"  {t['description']}")

## 8. Latency Optimization Techniques

In [ ]:
print("Latency Optimization Techniques")
print("=" * 60)

latency_techniques = [
    {
        "technique": "CUDA Graphs",
        "impact": "20-50% decode latency reduction",
        "when": "Decode phase (fixed shapes)",
        "how": "Eliminate kernel launch overhead",
    },
    {
        "technique": "Operator Fusion",
        "impact": "10-30% overall latency reduction",
        "when": "Always applicable",
        "how": "Fuse add+norm, QKV projection, gate+up projection",
    },
    {
        "technique": "FlashAttention",
        "impact": "2-4x attention speedup",
        "when": "Prefill phase (long sequences)",
        "how": "IO-aware tiling reduces memory bandwidth needs",
    },
    {
        "technique": "Tensor Parallelism",
        "impact": "~Linear latency reduction with # GPUs",
        "when": "Model doesn't fit on one GPU, or need low latency",
        "how": "Split compute across GPUs, all-reduce at sync points",
    },
    {
        "technique": "Quantization (FP8/INT4)",
        "impact": "1.5-3x decode latency reduction",
        "when": "Decode is memory-bandwidth bound",
        "how": "Reduce weight size -> faster memory reads",
    },
]

for t in latency_techniques:
    print(f"\n{t['technique']}:")
    print(f"  Impact: {t['impact']}")
    print(f"  When: {t['when']}")
    print(f"  How: {t['how']}")

## 9. A/B Performance Testing

In [ ]:
class PerformanceComparator:
    """
    A/B test two model implementations for performance.
    """
    
    def __init__(self, model_a, model_b, label_a="Baseline", label_b="Optimized"):
        self.model_a = model_a
        self.model_b = model_b
        self.label_a = label_a
        self.label_b = label_b
    
    def benchmark(self, model, input_tensor, n_warmup=10, n_runs=50):
        """Benchmark a model."""
        # Warmup
        for _ in range(n_warmup):
            with torch.no_grad():
                _ = model(input_tensor)
        
        # Measure
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            with torch.no_grad():
                _ = model(input_tensor)
            times.append((time.perf_counter() - start) * 1000)
        
        return np.array(times)
    
    def compare(self, input_configs: List[dict]):
        """
        Compare performance across different input configurations.
        """
        print(f"\nA/B Performance Test: {self.label_a} vs {self.label_b}")
        print("=" * 70)
        
        results = []
        
        for config in input_configs:
            seq_len = config.get('seq_len', 128)
            batch_size = config.get('batch_size', 1)
            vocab_size = config.get('vocab_size', 1000)
            
            input_tensor = torch.randint(0, vocab_size, (batch_size, seq_len))
            
            times_a = self.benchmark(self.model_a, input_tensor)
            times_b = self.benchmark(self.model_b, input_tensor)
            
            speedup = np.median(times_a) / np.median(times_b)
            
            result = {
                'config': f"batch={batch_size}, seq={seq_len}",
                'time_a': np.median(times_a),
                'time_b': np.median(times_b),
                'speedup': speedup,
            }
            results.append(result)
            
            direction = "faster" if speedup > 1 else "slower"
            print(f"  {result['config']:25s}  "
                  f"{self.label_a}: {result['time_a']:.2f}ms  "
                  f"{self.label_b}: {result['time_b']:.2f}ms  "
                  f"Speedup: {speedup:.2f}x ({direction})")
        
        # Summary
        avg_speedup = np.mean([r['speedup'] for r in results])
        print(f"\n  Average speedup: {avg_speedup:.2f}x")
        
        return results

# Demo A/B test
# Model A: Simple implementation
model_a = nn.Sequential(
    nn.Embedding(1000, 256),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 1000),
)

# Model B: "Optimized" (fused layers)
model_b = nn.Sequential(
    nn.Embedding(1000, 256),
    nn.Linear(256, 1000),  # Skip intermediate layer
)

comparator = PerformanceComparator(model_a, model_b, "Baseline", "Optimized")
results = comparator.compare([
    {'seq_len': 16, 'batch_size': 1, 'vocab_size': 1000},
    {'seq_len': 128, 'batch_size': 1, 'vocab_size': 1000},
    {'seq_len': 128, 'batch_size': 8, 'vocab_size': 1000},
    {'seq_len': 512, 'batch_size': 1, 'vocab_size': 1000},
])

## 10. Performance Checklist for New Models

In [ ]:
print("Performance Checklist for New Model Implementations")
print("=" * 60)

checklist = [
    # Linear layers
    ("Linear Layers", [
        "Use QKVParallelLinear for merged Q/K/V (single matmul)",
        "Use MergedColumnParallelLinear for gate+up (single matmul)",
        "Use RowParallelLinear for o_proj and down_proj",
        "Pass quant_config to enable quantized kernels",
    ]),
    # Attention
    ("Attention", [
        "Use vLLM's Attention or SGLang's RadixAttention wrapper",
        "Do NOT implement attention manually",
        "Ensure RoPE is applied before the attention wrapper",
        "Use the correct head_dim for GQA models",
    ]),
    # Norms and activations
    ("Norms & Activations", [
        "Use fused RMSNorm with residual (single kernel)",
        "Use SiluAndMul for fused SiLU + element-wise multiply",
        "Avoid creating intermediate tensors for norms",
    ]),
    # Memory
    ("Memory", [
        "Support GQA (fewer KV heads -> smaller KV cache)",
        "Handle tie_word_embeddings (share embed/lm_head weights)",
        "Use in-place operations where possible",
        "Avoid unnecessary .clone() or .contiguous() calls",
    ]),
    # CUDA graphs
    ("CUDA Graph Compatibility", [
        "No dynamic tensor shapes in decode path",
        "No CPU-GPU synchronization (.item(), .cpu())",
        "No Python control flow based on tensor values",
        "Pre-allocate all buffers",
    ]),
    # Tensor parallelism
    ("Tensor Parallelism", [
        "Use vLLM's parallel layers (ColumnParallel, RowParallel)",
        "Ensure num_heads and num_kv_heads are divisible by TP size",
        "Handle all-reduce communication correctly",
        "Test with TP=1, 2, 4, 8",
    ]),
]

for category, items in checklist:
    print(f"\n{category}:")
    for item in items:
        print(f"  [ ] {item}")

## 11. Exercises

### Exercise 1: Profile a Model
Create a transformer model and profile it at different sequence lengths. Identify when attention becomes the bottleneck.

### Exercise 2: Memory Budget Calculator
Build a calculator that, given a model config and GPU memory, determines the maximum batch size and sequence length.

### Exercise 3: A/B Test Optimization
Implement two versions of an MLP (standard vs fused SiLU+mul) and benchmark them.

In [ ]:
# Exercise 2: Memory Budget Calculator

def memory_budget_calculator(
    gpu_memory_gb: float,
    hidden_size: int,
    num_layers: int,
    vocab_size: int,
    intermediate_size: int,
    num_heads: int,
    num_kv_heads: int,
    dtype_bytes: int = 2,
    overhead_ratio: float = 0.1,
):
    """
    Calculate maximum batch size and sequence length given GPU memory.
    """
    gpu_bytes = gpu_memory_gb * (1024**3)
    usable_bytes = gpu_bytes * (1 - overhead_ratio)  # Reserve for overhead
    
    # Model weight memory
    model_mem = estimate_model_memory(
        hidden_size, num_layers, vocab_size,
        intermediate_size, num_heads, num_kv_heads, dtype_bytes
    )['total_bytes']
    
    remaining = usable_bytes - model_mem
    
    # KV cache memory per token
    head_dim = hidden_size // num_heads
    kv_per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes
    
    print(f"Memory Budget Calculator")
    print(f"========================")
    print(f"GPU Memory: {gpu_memory_gb} GB")
    print(f"Model Weights: {model_mem/1e9:.2f} GB")
    print(f"Available for KV Cache: {remaining/1e9:.2f} GB")
    print(f"KV Cache per token: {kv_per_token} bytes")
    print()
    
    # Calculate max tokens
    max_tokens = int(remaining / kv_per_token)
    
    print(f"Maximum total tokens in KV cache: {max_tokens:,}")
    print()
    print(f"Example configurations:")
    print(f"{'Batch Size':>12} {'Max Seq Len':>12} {'Total Tokens':>14}")
    print("-" * 40)
    for batch_size in [1, 4, 8, 16, 32, 64, 128, 256]:
        max_seq = max_tokens // batch_size
        if max_seq < 1:
            break
        print(f"{batch_size:>12} {max_seq:>12,} {batch_size * max_seq:>14,}")

# Llama-7B on an A100 80GB
memory_budget_calculator(
    gpu_memory_gb=80,
    hidden_size=4096,
    num_layers=32,
    vocab_size=32000,
    intermediate_size=11008,
    num_heads=32,
    num_kv_heads=32,
    dtype_bytes=2,  # FP16
)

In [ ]:
# Llama-7B on a consumer GPU (24GB)
print("\n" + "=" * 60)
print("Llama-7B on 24GB GPU (RTX 4090 / A5000):")
print("=" * 60)

memory_budget_calculator(
    gpu_memory_gb=24,
    hidden_size=4096,
    num_layers=32,
    vocab_size=32000,
    intermediate_size=11008,
    num_heads=32,
    num_kv_heads=32,
    dtype_bytes=2,  # FP16
)

print("\n" + "=" * 60)
print("Llama-7B GPTQ-4bit on 24GB GPU:")
print("=" * 60)

memory_budget_calculator(
    gpu_memory_gb=24,
    hidden_size=4096,
    num_layers=32,
    vocab_size=32000,
    intermediate_size=11008,
    num_heads=32,
    num_kv_heads=32,
    dtype_bytes=1,  # INT4 weights ~ 0.5 bytes, but KV cache stays FP16
)

## Summary

In this notebook, we covered:

1. **Profiling**: Use torch.profiler, CUDA events, nsight for bottleneck identification
2. **Compute Breakdown**: FFN dominates at short sequences, attention at long sequences
3. **Kernel Selection**: FlashAttention for prefill, FlashInfer for decode, Marlin for INT4
4. **Memory Optimization**: Fused ops, in-place ops, GQA, paged KV cache, quantization
5. **CUDA Graphs**: 20-50% decode latency reduction, requires fixed tensor shapes
6. **Operating Points**: Small batch = memory-bound, large batch = compute-bound
7. **Throughput**: Continuous batching, speculative decoding, prefix caching
8. **Latency**: CUDA graphs, operator fusion, FlashAttention, TP
9. **A/B Testing**: Systematic comparison with statistical rigor
10. **Checklist**: Comprehensive performance checklist for new models

### Key Takeaway
Use vLLM/SGLang's built-in layer primitives (parallel linears, fused norms, attention wrappers). They are already highly optimized. Your job as a model implementer is to wire them together correctly, and the performance will be good by default.